# Random Forest Modeling for Remaining CO2 Uptake Targets

This lightweight notebook evaluates the remaining pressure targets using the best Random Forest parameters selected previously. It uses one consistent 60/20/20 train-validation-test split for each target and avoids repeating the expensive grid search.

In [2]:
# Section 1: Imports and reproducibility setup
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

SEED = 12345

In [3]:
# Section 2: Features, targets, and fixed best RF parameters
feature_columns = ["lcd", "pld", "void_fraction", "surface_area_m2g"]
remaining_targets = [
    "CO2_uptake_0.1bar_molkg",
    "CO2_uptake_0.5bar_molkg",
    "CO2_uptake_2.5bar_molkg",
]

best_rf_params = {
    "n_estimators": 200,
    "max_depth": 15,
    "min_samples_split": 3,
    "min_samples_leaf": 4,
    "random_state": SEED,
    "n_jobs": -1,
}

### Where `random_state=SEED` Is Needed

`np.random.seed(SEED)` does not automatically control the random-number generators used internally by scikit-learn. Pass the seed directly to each stochastic operation so the results are reproducible.

| Operation or model | Add `random_state=SEED`? | Reason |
|---|---|---|
| `RandomForestRegressor` | **Yes** | Tree construction and feature sampling are stochastic. |
| `GradientBoostingRegressor` | **Yes** | Use it explicitly for reproducible model behavior and future configuration changes. |
| `train_test_split(..., shuffle=True)` | **Yes** | Keeps the train, validation, and test rows identical across runs. |
| `KFold(..., shuffle=True)` | **Yes** | Keeps the cross-validation folds identical across runs. |
| `LinearRegression` | No | Deterministic for the same data. |
| `Ridge` | No | Deterministic for the same data and configuration. |
| `DummyRegressor` | No | Deterministic for the same data and strategy. |

Use the same `SEED` for every stochastic model, data split, and shuffled cross-validation object when comparing results. In this notebook, the fixed Random Forest parameters include `random_state=SEED`, and the split function uses `random_state=seed` so it can be reused safely.

In [12]:
# Section 3: Load the processed dataset, keep modeling columns, and validate them
data_path = Path("/home/susan/mof-co2-adsorption/data/processed/data_clean_v2")
df = pd.read_csv(data_path)

required_columns = feature_columns + remaining_targets
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(
        "Missing required columns: " + ", ".join(missing_columns)
    )

# Keep only the four features and three remaining targets needed in this notebook.
df = df[required_columns].copy()

print(f"Reduced dataset shape: {df.shape}")
print("Columns used:", list(df.columns))

Reduced dataset shape: (31234, 7)
Columns used: ['lcd', 'pld', 'void_fraction', 'surface_area_m2g', 'CO2_uptake_0.1bar_molkg', 'CO2_uptake_0.5bar_molkg', 'CO2_uptake_2.5bar_molkg']


In [13]:
# Section 4: Reusable 60/20/20 train-validation-test split
def split_target_data(df, feature_columns, target_column, seed=SEED):
    """Return reproducible 60/20/20 splits for one target."""
    selected_features = df[feature_columns].copy()
    selected_target = df[target_column].copy()

    X_train, X_temp, y_train, y_temp = train_test_split(
        selected_features,
        selected_target,
        test_size=0.4,
        shuffle=True,
        random_state=seed,
    )
    X_valid, X_test, y_valid, y_test = train_test_split(
        X_temp,
        y_temp,
        test_size=0.5,
        shuffle=True,
        random_state=seed,
    )
    return X_train, X_valid, X_test, y_train, y_valid, y_test

X_train, X_valid, X_test, y_train, y_valid, y_test = split_target_data(
    df, feature_columns, remaining_targets[0]
 )
print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_valid.shape, y_valid.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (18740, 4) (18740,)
Validation: (6247, 4) (6247,)
Test: (6247, 4) (6247,)


In [14]:
# Section 5: Reusable Random Forest training and evaluation
def train_and_evaluate_random_forest(
    X_train, X_valid, X_test, y_train, y_valid, y_test, rf_params
 ):
    """Fit one RF model and return metrics and predictions for all splits."""
    model = RandomForestRegressor(**rf_params)
    model.fit(X_train, y_train)

    predictions = {
        "train": model.predict(X_train),
        "valid": model.predict(X_valid),
        "test": model.predict(X_test),
    }
    actual_values = {
        "train": y_train,
        "valid": y_valid,
        "test": y_test,
    }

    metrics = {}
    for split_name, y_true in actual_values.items():
        y_pred = predictions[split_name]
        metrics[split_name] = {
            "MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
            "R2": r2_score(y_true, y_pred),
        }

    return {
        "model": model,
        "metrics": metrics,
        "predictions": predictions,
    }

In [15]:
# Section 6: Run the fixed-parameter RF for each remaining target
target_results = {}
summary_rows = []

for target_column in remaining_targets:
    splits = split_target_data(df, feature_columns, target_column)
    evaluation = train_and_evaluate_random_forest(*splits, best_rf_params)
    target_results[target_column] = evaluation

    row = {"target": target_column}
    for split_name, split_metrics in evaluation["metrics"].items():
        for metric_name, metric_value in split_metrics.items():
            row[f"{split_name}_{metric_name}"] = metric_value
    row["train_valid_R2_gap"] = row["train_R2"] - row["valid_R2"]
    row["valid_test_R2_gap"] = row["valid_R2"] - row["test_R2"]
    summary_rows.append(row)

summary_table = pd.DataFrame(summary_rows)
summary_table

,target,train_MAE,train_RMSE,train_R2,valid_MAE,valid_RMSE,valid_R2,test_MAE,test_RMSE,test_R2,train_valid_R2_gap,valid_test_R2_gap
0,CO2_uptake_0.1bar_molkg,0.227258,0.357411,0.767992,0.317822,0.489274,0.576851,0.316986,0.492945,0.581790,0.191141,-0.004940
1,CO2_uptake_0.5bar_molkg,0.485365,0.676513,0.779288,0.677490,0.932366,0.604088,0.670211,0.916177,0.619096,0.175200,-0.015008
2,CO2_uptake_2.5bar_molkg,0.859933,1.175651,0.786497,1.136638,1.515256,0.656750,1.145255,1.532488,0.638080,0.129747,0.018670


In [16]:
# Section 7: Order the summary table by pressure
pressure_order = {target: index for index, target in enumerate(remaining_targets)}
summary_table = (
    summary_table.assign(_pressure_order=summary_table["target"].map(pressure_order))
    .sort_values("_pressure_order")
    .drop(columns="_pressure_order")
    .reset_index(drop=True)
 )
summary_table

,target,train_MAE,train_RMSE,train_R2,valid_MAE,valid_RMSE,valid_R2,test_MAE,test_RMSE,test_R2,train_valid_R2_gap,valid_test_R2_gap
0,CO2_uptake_0.1bar_molkg,0.227258,0.357411,0.767992,0.317822,0.489274,0.576851,0.316986,0.492945,0.581790,0.191141,-0.004940
1,CO2_uptake_0.5bar_molkg,0.485365,0.676513,0.779288,0.677490,0.932366,0.604088,0.670211,0.916177,0.619096,0.175200,-0.015008
2,CO2_uptake_2.5bar_molkg,0.859933,1.175651,0.786497,1.136638,1.515256,0.656750,1.145255,1.532488,0.638080,0.129747,0.018670


## Interpretation of Remaining Target Results

The fixed-parameter Random Forest was evaluated for CO2 uptake at 0.1, 0.5, and 2.5 bar using the same reproducible 60/20/20 split procedure.

| Target | Validation MAE | Validation RMSE | Validation R2 | Test MAE | Test RMSE | Test R2 |
|---|---:|---:|---:|---:|---:|---:|
| 0.1 bar | 0.317822 | 0.489274 | 0.576851 | 0.316986 | 0.492945 | 0.581790 |
| 0.5 bar | 0.677490 | 0.932366 | 0.604088 | 0.670211 | 0.916177 | 0.619096 |
| 2.5 bar | 1.136638 | 1.515256 | 0.656750 | 1.145255 | 1.532488 | 0.638080 |

### Training and Validation Performance

For all three targets, the training errors are lower than the validation errors and the training R2 values are higher than the validation R2 values. This is the expected pattern when the model fits the training data better than unseen data and indicates some overfitting.

The training-to-validation R2 gaps are:

- 0.1 bar: **0.191141**
- 0.5 bar: **0.175200**
- 2.5 bar: **0.129747**

The gap decreases as pressure increases. This suggests that the model's generalization is relatively more consistent at 2.5 bar, although the gap alone should not be used as the only model-selection criterion.

### Validation and Test Generalization

The validation and test results are close for all three targets:

- At **0.1 bar**, test R2 is **0.581790**, slightly higher than validation R2 of 0.576851. Test MAE is nearly unchanged, while test RMSE is slightly higher.
- At **0.5 bar**, test R2 is **0.619096**, higher than validation R2 of 0.604088. Test MAE and RMSE are slightly lower than validation values.
- At **2.5 bar**, test R2 is **0.638080**, lower than validation R2 of 0.656750. Test MAE and RMSE are slightly higher than validation values.

These differences do not indicate a major validation-to-test performance collapse. They show that the validation and test subsets have somewhat different difficulty for each pressure. The results should still be interpreted cautiously because they come from one train/validation/test split.

### Pressure Comparison and Feature Interpretation

The test R2 of **0.638080** at 2.5 bar means that the model explains approximately **63.8% of the variation in CO2 uptake values** for unseen test MOFs at that pressure. It does not mean that the four features account for 63.8% of the amount of CO2 adsorbed, nor does it establish a causal relationship.

The model uses four geometric descriptors: LCD, PLD, void fraction, and surface area. Together, these features provide stronger predictive information at 2.5 bar than at 0.1 bar in this evaluation, as test R2 increases from **0.581790** to **0.638080**. A reasonable interpretation is that geometric pore properties are more useful for predicting relative uptake differences at higher pressure, where pore volume, accessible space, pore dimensions, and surface area may play a larger role in the observed uptake patterns.

This interpretation is predictive rather than causal. The model does not show which individual feature is responsible for the increase, and the higher R2 may also reflect the distribution and scale of the target at each pressure. Feature importance or permutation importance would be needed to determine which descriptors contribute most at each pressure.

The absolute MAE and RMSE increase with pressure because the CO2 uptake values also become larger. Therefore, MAE and RMSE should mainly be compared between models at the same pressure, not directly across pressures.

The results suggest that the four geometric descriptors provide moderate predictive signal across the remaining pressure targets, with the strongest explained variation at 2.5 bar in this evaluation. The remaining error may reflect missing chemical and structural information such as framework composition, functional groups, charge-related properties, topology, and adsorption-site effects.

### Practical Conclusion

The fixed-parameter Random Forest is suitable for preliminary screening and ranking of MOFs across the remaining pressure targets. It is not sufficiently precise for exact uptake prediction for individual MOFs. The model should be reported with the pressure-specific MAE, RMSE, and R2 values, and the results should be confirmed with repeated cross-validation or additional splits before making stronger generalization claims.

In [19]:
# Section 8: Create per-target test prediction breakdowns
prediction_breakdowns = {}

for target_column in remaining_targets:
    evaluation = target_results[target_column]
    predictions = evaluation["predictions"]["test"]
    _, _, X_test_target, _, _, y_test_target = split_target_data(
        df, feature_columns, target_column
    )
    breakdown = pd.DataFrame({
        "y_true_test": y_test_target.to_numpy(),
        "y_pred_test": predictions,
    })
    breakdown["residual"] = breakdown["y_true_test"] - breakdown["y_pred_test"]
    breakdown["absolute_error"] = breakdown["residual"].abs()
    prediction_breakdowns[target_column] = breakdown

prediction_breakdowns[remaining_targets[0]].head()

,y_true_test,y_pred_test,residual,absolute_error
0,1.258330,2.068833,-0.810503,0.810503
1,0.338780,0.266250,0.072530,0.072530
2,0.457410,0.451854,0.005556,0.005556
3,0.179677,0.563047,-0.383370,0.383370
4,0.142646,0.513665,-0.371019,0.371019


### How to Interpret the Test Prediction Breakdown

This table compares the actual CO2 uptake values with the values predicted by the Random Forest for the selected test MOFs.

| Column | Meaning |
|---|---|
| `y_true_test` | Observed CO2 uptake value in the test set. |
| `y_pred_test` | CO2 uptake predicted by the trained Random Forest. |
| `residual` | Signed prediction error, calculated as `y_true_test - y_pred_test`. |
| `absolute_error` | Size of the prediction error without its direction. |

A positive residual means that the model underestimated the observed uptake. A negative residual means that the model overestimated the observed uptake. Smaller absolute-error values indicate more accurate predictions for individual MOFs.

The table is useful for identifying individual materials with relatively large errors. However, the overall MAE, RMSE, and R2 values in the summary table should be used to judge model performance across the complete test set.

In [20]:
# Section 9: Save metrics and prediction breakdowns
results_directory = Path("/home/susan/mof-co2-adsorption/results/remaining_targets")
results_directory.mkdir(parents=True, exist_ok=True)

summary_path = results_directory / "remaining_targets_rf_metrics.csv"
summary_table.to_csv(summary_path, index=False)

for target_column, breakdown in prediction_breakdowns.items():
    pressure_name = target_column.removeprefix("CO2_uptake_").removesuffix("_molkg")
    breakdown_path = results_directory / f"{pressure_name}_rf_test_predictions.csv"
    breakdown.to_csv(breakdown_path, index=False)

print(f"Saved summary to: {summary_path}")
print(f"Saved {len(prediction_breakdowns)} prediction breakdown files.")

Saved summary to: /home/susan/mof-co2-adsorption/results/remaining_targets/remaining_targets_rf_metrics.csv
Saved 3 prediction breakdown files.
